### Import modules and utility code

In [1]:
import os
import flatdict as fd
import numpy as np
import yaml
import pint
import h5py
import hashlib
from pint import UnitRegistry

ureg = UnitRegistry()
# ureg.formatter.default_format = "D"
# https://pint.readthedocs.io/en/stable/user/formatting.html

ModuleNotFoundError: No module named 'flatdict'

In [ ]:
def recursive_assign(path, idx, leaf_idx, obj, nested_dict):
    if -1 < idx < leaf_idx:
        if path[idx] not in nested_dict:
            # print(f"\tgrp {level}")
            # nested_dict[f"{'/'.join(path[0:idx])}"] = {}
            nested_dict[path[idx]] = {}
        recursive_assign(path, idx + 1, leaf_idx, obj, nested_dict[path[idx]])
    else:
        if path[idx] not in nested_dict:
            # print(f"\t\tdset {level}")
            if isinstance(obj, ureg.Quantity):  # TODO fix
                if path[idx] not in nested_dict:
                    nested_dict[path[idx]] = {}
                    if obj.dimensionless:
                        if np.shape(obj.magnitude) == ():
                            nested_dict[path[idx]]["value"] = (
                                obj.magnitude.item()
                            )  # ["value"] value only for consistency
                        else:
                            nested_dict[path[idx]]["value"] = obj.magnitude.tolist()
                        if obj.units == "radian":
                            nested_dict[path[idx]]["@units"] = f"{obj.units}"
                    else:
                        if np.shape(obj.magnitude) == ():
                            nested_dict[path[idx]]["value"] = (
                                obj.magnitude.item()
                            )  # ["value"] value only for consistency
                        else:
                            nested_dict[path[idx]]["value"] = obj.magnitude.tolist()
                        nested_dict[path[idx]]["@units"] = f"{obj.units}"
            elif isinstance(obj, (str, bool)):
                if path[idx] not in nested_dict:
                    # nested_dict[path[idx]] = {}
                    nested_dict[path[idx]] = obj  # ["value"] value only for consistency
            elif isinstance(obj, list):
                if all(True if isinstance(val, str) else False for val in obj):
                    # if path[idx] not in nested_dict:
                    #     nested_dict[path[idx]] = {}
                    nested_dict[path[idx]] = obj  # ["value"] value only for consistency
    return nested_dict

In [ ]:
def get_sha256_of_file_content(fpath) -> str:
    """Compute a SHA256 hashvalue of given file."""
    fh = hashlib.sha256()
    with open(fpath, "rb") as fp:
        while True:
            chunk = fp.read(fh.block_size)
            if not chunk:
                break
            fh.update(chunk)
    return f"{fh.hexdigest()}"

### Create a configuration file for SCORE following the NXmicrostructure_score_config.nxdl.xml contributed application definition

<div class="alert alert-block alert-info">Modify those values in this populated template that match your case.</div>

In [ ]:
simid = 1


class ScoreConfig:
    # attributes use @ as a marker/escape character
    # groups are all those concepts which have an attribute @NX_class
    # fields are all those other concepts with an attribute %value
    def __init__(self, simid):
        self.deg2kelvin = 273.15
        self.sim_id = simid
        self.entry_id = 1
        self.yaml_file_name = f"NXmicrostructure_score_config.SimID.{self.sim_id}.yaml"
        self.hdf5_file_name = f"SCORE.Config.SimID.{self.sim_id}.nxs"
        self.cfg: Dict[str, Any] = {}
        self.cfg["@NX_class"] = "NXroot"
        self.cfg["@default"] = f"entry{self.entry_id}"
        self.cfg[f"entry{self.entry_id}/@NX_class"] = "NXentry"
        self.cfg[f"entry{self.entry_id}/@default"] = "time_temperature"
        self.cfg[f"entry{self.entry_id}/definition"] = "NXmicrostructure_score_config"
        # "simulation_identifier": None,
        # "start_time": None,
        # "end_time": None,
        # "program1": None
        # physics
        self.description()
        self.user()
        self.sample()
        self.material()
        self.deformation()
        self.nucleation()
        self.mobility()
        self.recovery()
        self.drag()
        self.component_analysis()
        self.time_temperature()
        # numerics
        self.discretization()
        self.numerics()
        self.solitary_unit_model()

    def description(self):
        """Configure ELN-like description."""
        trg = f"entry{self.entry_id}/"
        self.cfg[f"{trg}description"] = (
            "Example showing how to bring recrystallization simulations with SCORE into NOMAD."
            ""
        )

    def user(self):
        """Configure the user."""
        for idx, name in enumerate(["MarkusK", "ViteshS"]):
            trg = f"entry{self.entry_id}/user{idx + 1}/"
            self.cfg[f"{trg}@NX_class"] = "NXuser"
            self.cfg[f"{trg}name"] = name

    def sample(self):
        """Configure contextualizing details about the sample."""
        trg = f"entry{self.entry_id}/sample/"
        self.cfg[f"{trg}@NX_class"] = "NXsample"
        self.cfg[f"{trg}dimensionality"] = ureg.Quantity(np.uint32(3))
        self.cfg[f"{trg}type"] = "simulation"
        self.cfg[f"{trg}atom_types"] = "Al"

    def material(self):
        """Configure material model single-phase only."""
        trg = f"entry{self.entry_id}/material/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        self.cfg[f"{trg}shear_modulus"] = ureg.Quantity(np.float64(9.5e9), ureg.pascal)
        self.cfg[f"{trg}burgers_vector"] = ureg.Quantity(
            np.float64(0.286), ureg.nanometer
        )
        self.cfg[f"{trg}melting_temperature"] = ureg.Quantity(
            np.float64(800.0 + self.deg2kelvin), ureg.kelvin
        )

    def deformation(self):
        """Configure deformed (initial) microstructure."""
        trg = f"entry{self.entry_id}/deformation/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        self.cfg[f"{trg}model"] = "damask"
        self.cfg[f"{trg}damask/@NX_class"] = "NXnote"
        damask_file_name = (
            "Tphase_Vf_01_tensionX_remeshed_increment19000_segmented.dream3D"
        )
        self.cfg[f"{trg}damask/file_name"] = (
            f"{damask_file_name}"  # how the score application sees this file
        )
        # given that the application is in score_root/build but the examples in score_root/examples/damask
        self.cfg[f"{trg}damask/algorithm"] = "sha256"
        # from the perspective of this JupyterNotebook's current working directory (e.g. check os.getcwd())
        self.cfg[f"{trg}damask/checksum"] = get_sha256_of_file_content(
            f"../../examples/damask/{damask_file_name}"
        )

    def nucleation(self):
        """Configure nucleation model."""
        trg = f"entry{self.entry_id}/nucleation/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        self.cfg[f"{trg}spatial_distribution"] = "damask"
        self.cfg[f"{trg}number_of_nuclei"] = ureg.Quantity(np.uint32(1))
        self.cfg[f"{trg}incubation_time"] = "site_saturation"
        # self.cfg[f"{trg}incubation_time_sigma"] = ureg.Quantity(np.float64(1000.), ureg.second)
        self.cfg[f"{trg}orientation"] = "damask"
        self.cfg[f"{trg}ensemble/@NX_class"] = "NXobject"
        self.cfg[f"{trg}ensemble/bunge_euler"] = ureg.Quantity(
            np.asarray([[0.0, 30.0, 45.0]], np.float64) / 180.0 * np.pi, ureg.radian
        )
        self.cfg[f"{trg}ensemble/incubation_time"] = ureg.Quantity(
            np.asarray([0.0], np.float64), ureg.second
        )

    def mobility(self):
        """Configure grain boundary mobility model."""
        trg = f"entry{self.entry_id}/grain_boundary_mobility/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        model = "rollett_holm"  # "sebald_gottstein"
        if model == "sebald_gottstein":
            self.cfg[f"{trg}@NX_class"] = "NXobject"
            self.cfg[f"{trg}model"] = model
            trg += "sebald_gottstein/"
            self.cfg[f"{trg}@NX_class"] = "NXobject"
            self.cfg[f"{trg}lagb_to_hagb_cut"] = ureg.Quantity(
                np.float64(15.0) / 180.0 * np.pi, ureg.radian
            )
            self.cfg[f"{trg}lagb_pre_factor"] = ureg.Quantity(
                np.float64(30.0), ureg.meter**4 / (ureg.joule * ureg.second)
            )
            self.cfg[f"{trg}lagb_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            self.cfg[f"{trg}hagb_pre_factor"] = ureg.Quantity(
                np.float64(30.0), ureg.meter**4 / (ureg.joule * ureg.second)
            )
            self.cfg[f"{trg}hagb_enthalpy"] = ureg.Quantity(np.float64(2.45), ureg.eV)
            self.cfg[f"{trg}special_disorientation"] = ureg.Quantity(
                np.float64(10.0) / 180.0 * np.pi, ureg.radian
            )
            self.cfg[f"{trg}special_pre_factor"] = ureg.Quantity(
                np.float64(30.0), ureg.meter**4 / (ureg.joule * ureg.second)
            )
            self.cfg[f"{trg}special_enthalpy"] = ureg.Quantity(
                np.float64(2.45), ureg.eV
            )
            # MAXDISORI_TO_40DEG111
        elif model == "rollett_holm":
            self.cfg[f"{trg}model"] = model
            trg += "rollett_holm/"
            self.cfg[f"{trg}@NX_class"] = "NXobject"
            self.cfg[f"{trg}m_null"] = ureg.Quantity(
                np.float64(300.0), ureg.meter**4 / (ureg.joule * ureg.second)
            )
            self.cfg[f"{trg}enthalpy"] = ureg.Quantity(np.float64(2.0314), ureg.eV)
            self.cfg[f"{trg}c1"] = ureg.Quantity(np.float64(0.9))
            self.cfg[f"{trg}c2"] = ureg.Quantity(np.float64(5.0))
            self.cfg[f"{trg}c3"] = ureg.Quantity(np.float64(9.0))

    def recovery(self):
        """Configure stored elastic energy recovery model."""
        trg = f"entry{self.entry_id}/stored_energy_recovery/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        self.cfg[f"{trg}model"] = "none"

    def drag(self):
        """Configure reduction of grain boundary mobility by dispersoids (Zener-Smith drag)."""
        trg = f"entry{self.entry_id}/dispersoid_drag/"
        self.cfg[f"{trg}@NX_class"] = "NXdata"
        self.cfg[f"{trg}model"] = "none"
        # trg = "entry{self.entry_id}/dispersoid_drag/zener_smith/"
        # self.cfg[f"{trg}@NX_class"] = "NXobject"
        # self.cfg[f"{trg}@signal"] = "radius"
        # self.cfg[f"{trg}@axes"] = "time"
        # self.cfg[f"{trg}@time_indices"] = ureg.Quantity(np.uint32(0))
        # self.cfg[f"{trg}@radius_indices"] = ureg.Quantity(np.uint32(1))
        # self.cfg[f"{trg}title"] = "r(t)"
        # self.cfg[f"{trg}pre_factor"] = ureg.Quantity(np.float64(1.5))
        # self.cfg[f"{trg}surface_energy"] = ureg.Quantity(np.float64(0.324), ureg.joule / (ureg.meter ** 2))
        # self.cfg[f"{trg}time/@long_name"] = "t (s)"
        # self.cfg[f"{trg}radius/@long_name"] = "r (nm)"
        # self.cfg[f"{trg}time"] = ureg.Quantity(np.asarray([0., 3600.], np.float64), ureg.second)
        # self.cfg[f"{trg}radius"] = ureg.Quantity(np.asarray([0., 0.], np.float64), ureg.nanometer)

    def component_analysis(self):
        """Configure which macrotexture components the code should bookkeep for volume evolution."""
        trg = f"entry{self.entry_id}/component_analysis/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        self.cfg[f"{trg}name"] = ["Alpha001-110", "Alpha112-110"]
        self.cfg[f"{trg}bunge_euler"] = ureg.Quantity(
            np.asarray([[0.0, 0.0, 0.0], [0.0, 35.0, 45.0]], np.float64)
            / 180.0
            * np.pi,
            ureg.radian,
        )
        self.cfg[f"{trg}theta"] = ureg.Quantity(
            np.asarray([10.0, 10.0], np.float64) / 180.0 * np.pi, ureg.radian
        )

    def time_temperature(self):
        """Configure the simulated annealing schedule."""
        trg = f"entry{self.entry_id}/time_temperature/"
        self.cfg[f"{trg}@NX_class"] = "NXdata"
        self.cfg[f"{trg}@signal"] = "temperature"
        self.cfg[f"{trg}@axes"] = "time"
        self.cfg[f"{trg}@time_indices"] = ureg.Quantity(np.uint32(0))
        self.cfg[f"{trg}@temperature_indices"] = ureg.Quantity(np.uint32(1))
        self.cfg[f"{trg}title"] = "T(t)"
        self.cfg[f"{trg}time/@long_name"] = "t (s)"
        self.cfg[f"{trg}temperature/@long_name"] = "T (K)"
        self.cfg[f"{trg}time"] = ureg.Quantity(
            np.asarray(
                [
                    0.0,
                    5.8,
                    100.8,
                    103.7,
                    105.0,
                    200.8,
                    205.8,
                    208.7,
                    300.0,
                    305.8,
                    405.8,
                    408.7,
                    600.0,
                ],
                np.float64,
            ),
            ureg.second,
        )
        self.cfg[f"{trg}temperature"] = ureg.Quantity(
            np.asarray(
                [
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                    500.0,
                    500.0,
                    20.0,
                    20.0,
                ],
                np.float64,
            )
            + self.deg2kelvin,
            ureg.kelvin,
        )

    def discretization(self):
        """Configure microstructure instantiation."""
        trg = f"entry{self.entry_id}/discretization/"
        self.cfg[f"{trg}@NX_class"] = "NXmicrostructure"
        trg += "grid/"
        self.cfg[f"{trg}@NX_class"] = "NXcg_grid"
        self.cfg[f"{trg}extent"] = ureg.Quantity(np.asarray([128, 52, 64], np.uint32))
        self.cfg[f"{trg}cell_dimensions"] = ureg.Quantity(
            np.float64(0.4982421), ureg.micrometer
        )

    def numerics(self):
        """Configure stop criteria and numerics."""
        trg = f"entry{self.entry_id}/numerics/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        self.cfg[f"{trg}max_x"] = ureg.Quantity(np.float64(1.0))
        self.cfg[f"{trg}max_time"] = ureg.Quantity(np.float64(3600.0), ureg.second)
        self.cfg[f"{trg}max_iteration"] = ureg.Quantity(np.uint32(10000))
        self.cfg[f"{trg}max_delta_x"] = ureg.Quantity(np.float64(0.1))
        self.cfg[f"{trg}x_set"] = ureg.Quantity(
            np.asarray(np.linspace(0.0, stop=1.0, num=101, endpoint=True), np.float64)
        )
        # self.cfg[f"{trg}percolation"] = False
        self.cfg[f"{trg}cell_cache/@NX_class"] = "NXobject"
        self.cfg[f"{trg}cell_cache/initial"] = ureg.Quantity(np.float64(0.2))
        self.cfg[f"{trg}cell_cache/realloc"] = ureg.Quantity(np.float64(1.0))
        self.cfg[f"{trg}cell_cache/defragment"] = True
        self.cfg[f"{trg}cell_cache/defragment_x"] = ureg.Quantity(
            np.asarray(
                [
                    0.60,
                    0.65,
                    0.70,
                    0.75,
                    0.80,
                    0.85,
                    0.90,
                    0.92,
                    0.94,
                    0.96,
                    0.98,
                    0.99,
                    0.999,
                    0.9999,
                ],
                np.float64,
            )
        )

    def solitary_unit_model(self):
        """Configure solitary unit post-processing approach."""
        trg = f"entry{self.entry_id}/solitary_unit/"
        self.cfg[f"{trg}@NX_class"] = "NXobject"
        self.cfg[f"{trg}apply"] = False
        self.cfg[f"{trg}number_of_domains"] = ureg.Quantity(np.uint32(1))
        self.cfg[f"{trg}rediscretization"] = ureg.Quantity(np.uint32(10000))
        # self.cfg[f"{trg}domain_identifier"] = None

    def write_config_yaml_file(self):
        """Write YAML config file to be used in SCORE."""
        template = {}
        for key, obj in self.cfg.items():
            path = key.split("/")
            recursive_assign(path, 0, len(path) - 1, obj, template)

        with open(self.yaml_file_name, "w") as fp:
            yaml.dump(template, fp, default_flow_style=None, sort_keys=False)
        print(f"{self.yaml_file_name} written")

    def get_flat_list_of_hfive_concept_paths(self):
        """Write HDF5 file to be used in NOMAD."""
        if os.path.isfile(self.yaml_file_name):
            with open(self.yaml_file_name, "r", encoding="utf-8") as stream:
                for key, val in fd.FlatDict(
                    yaml.safe_load(stream), delimiter="/"
                ).items():
                    # print(f"{key}, {val}")
                    if key.endswith("value"):
                        print(f"/{key[0:-6]}: {val}")
                    else:
                        print(f"/{key}: {val}")

    def write_config_hfive_file(self):
        with h5py.File(self.hdf5_file_name, "w") as h5w:
            sorted_flattened_dict = sorted(fd.FlatDict(self.cfg, delimiter="/").items())
            # for key, obj in sorted_flattened_dict:
            #     print(key)
            # instantiate groups and their attributes
            for key, obj in sorted_flattened_dict:
                if key.endswith("@NX_class") or key.endswith("@default"):
                    hpath = f"/{key[0 : key.rfind('@')]}"
                    attr_name = f"{key[key.rfind('@') + 1 :]}"
                    # print(f"group >>>> {hpath}, {attr_name}")
                    if hpath != "/":
                        grpnm = hpath[0:-1]
                        if grpnm not in h5w:
                            grp = h5w.create_group(grpnm)
                            grp.attrs[attr_name] = obj
                        else:
                            h5w[grpnm].attrs[attr_name] = obj
                    else:
                        h5w.attrs[attr_name] = obj
            # instantiate fields which is possible because groups now exist and
            # the h5py library internally assures that the data end up written
            # in that group mentioned because making e.g. create_dataset calls
            # can use even deeply nested paths on empty hdf5 and will recursively
            # add all those missing groups along the path from the leaf upwards to root
            for key, obj in sorted_flattened_dict:
                if "@" not in key:  # not key.endswith("@NX_class"):
                    hpath = f"/{key}"
                    # print(f"field >>>> {hpath}")
                    # print(f"{key}, {hpath}, {type(obj)}")
                    if isinstance(obj, ureg.Quantity):  # TODO fix
                        if obj.dimensionless:
                            if np.shape(obj.magnitude) == ():
                                dst = h5w.create_dataset(
                                    hpath, data=obj.magnitude.item()
                                )
                            else:
                                dst = h5w.create_dataset(
                                    hpath, data=obj.magnitude
                                )  # .tolist()
                            if obj.units == "radian":
                                dst.attrs["units"] = f"{obj.units}"
                        else:
                            if np.shape(obj.magnitude) == ():
                                dst = h5w.create_dataset(
                                    hpath, data=obj.magnitude.item()
                                )
                            else:
                                dst = h5w.create_dataset(
                                    hpath, data=obj.magnitude
                                )  # .tolist()
                            dst.attrs["units"] = f"{obj.units}"
                    elif isinstance(obj, (str, bool)):
                        dst = h5w.create_dataset(hpath, data=obj)
                    elif isinstance(obj, list):
                        if all(True if isinstance(val, str) else False for val in obj):
                            dst = h5w.create_dataset(hpath, data=obj)
            # add attributes to existent fields and groups which are not NX_class
            for key, obj in sorted_flattened_dict:
                if ("@" in key) and (not key.endswith("@NX_class")):
                    hpath = f"/{key[0 : key.rfind('@')]}"
                    if hpath != "//":
                        attr_name = f"{key[key.rfind('@') + 1 :]}"
                        # print(f"attribute >>>> {hpath}, {attr_name}")
                        h5w[hpath].attrs[attr_name] = obj
            print(f"{self.hdf5_file_name} written")

In [ ]:
yml = ScoreConfig(simid)
yml.write_config_yaml_file()
yml.get_flat_list_of_hfive_concept_paths()
yml.write_config_hfive_file()